In [1]:
# I2C 통신을 위한 통신 버스 모듈 추가
import busio
import board

WARNNIG: Jetson.GPIO library has not been verified with this carrier board,


In [2]:
import libraries.Omniwheel_Protocol as Omniwheel_Protocol
import serial
import time

In [3]:
# I2C PWM 제어 칩 전용 모듈 추가
import adafruit_pca9685

In [4]:
# I2C CLCD 제어 모듈 추가
import libraries.I2C_Charactor_Liquid_Crystal_Display as i2c_lcd

In [5]:
# 시간 지연을 위한 time 모듈 추가
import time

In [6]:
# I2C 통신을 이용하기 위해 busio의 I2C 클래스 객체 생성
i2c = busio.I2C(board.SCL, board.SDA)

In [7]:
# PCA9685 칩을 제어하기 위해 클래스 객체 생성
pca9685 = adafruit_pca9685.PCA9685(i2c)

# Duty Cycle 값 초기화
pca9685.channels[15].duty_cycle = 0

In [8]:
#          도   레   미   파   솔   라   시   도
melody = [130, 146, 164, 175, 196, 220, 247, 262]

In [9]:
# LCD를 제어하기 위해 클래스 객체 생성
lcd = i2c_lcd.LCD(i2c)

In [10]:
def display_lcd(text):
    lcd.clear()
    
    lcd.setCursor(0, 0)
    lcd.display_string(" Ultrasonic Low ")

    lcd.setCursor(1, 0)
    lcd.display_string(text)

In [11]:
Arduino_ID = 0x20
REQUEST_Ultra_Sonic_SENSOR = 0xA1
ANSWER_Ultra_Sonic_SENSOR = 0xB1
MID_Ultra_Sonic_SENSOR =0x80

In [12]:

CONTROL_DRIVE = 0xC0

MID_CONTROL_X_LINEAR_VELOCITY = 0x80
MID_CONTROL_Y_LINEAR_VELOCITY = 0x81
MID_CONTROL_W_ANGULAR_VELOCITY = 0x82

MID_list=[MID_CONTROL_X_LINEAR_VELOCITY,
          MID_CONTROL_Y_LINEAR_VELOCITY,
          MID_CONTROL_W_ANGULAR_VELOCITY
        ]


In [13]:
# -600 ~ 600
Moter_Control= [ 600,
                 0,
                 -600,
                 0
                ] 

In [14]:
send_packet = Omniwheel_Protocol.Packet()
recv_packet = Omniwheel_Protocol.Packet()

In [15]:
recv_packet.clearPacket()
send_packet.clearPacket()

In [16]:
recv_list = []
recv_parsing_packet = []

In [17]:
send_flag=False

In [18]:
Serial_Arduino = serial.Serial(port ="/dev/ttyACM0" ,baudrate = 115200, timeout=.1)
time.sleep(1)
print("connect complete")

connect complete


In [19]:
def Packet_send(_id, _cmd, _mid, _data = None):
    # 전역변수 사용
    global send_flag
    
    # 전송 완료 플래그 확인 
    if(send_flag==False):
        
        # 초기화
        send_packet.clearPacket()
        
        # ID 설정
        send_packet.setID(_id)
        
        # CMD 설정
        send_packet.setCMD(_cmd)
        
        # Payload 초기화
        send_packet.clearPayload()
        
        # MID, data 설정
        send_packet.addPayload(_mid, _data)
        
        # 패킷 LRC 계산
        send_packet.calcLRC_Lower()
        
        # 패킷을 리스트로 변환
        send_list=send_packet.packetToList()
        
        # Arduino 에 패킷 리스트 전송
        Serial_Arduino.write(send_list)
        
        # 전송 완료 플래그 설정
        send_flag=True

In [20]:
def Control_send(_id, _cmd, _mid, _data):
    # 패킷을 초기화 합니다.
    send_packet.clearPacket()

    # ID 설정 합니다.
    send_packet.setID(_id)

    # CMD 설정 합니다.
    send_packet.setCMD(_cmd)

    # Payload 초기화 합니다.
    send_packet.clearPayload()

    # MID, data 설정 합니다.
    send_packet.addPayload(_mid, _data)

    # 패킷 LRC 계산합니다.
    send_packet.calcLRC_Lower()

    # 패킷을 리스트로 변환합니다.
    send_list=send_packet.packetToList()
    
    # Arduino 에 패킷 리스트를 전송합니다.
    Serial_Arduino.write(send_list)

In [21]:
def Packet_receive(ser):
    # 전역변수 사용
    global send_flag
    
    # 전송 완료 플래그 확인
    if(send_flag==True):
        
        # 수신받은 데이터가 없을때까지
        while ser.inWaiting() > 0:
            
            # 1바이트씩 데이터를 받음
            Arduino_Data = ser.read(1)
            
            # 데이터를 받은 경우
            if(len(Arduino_Data)>0):
                
                # 수신 패킷 리스트에 수신 데이터 저장
                recv_list.append(ord(Arduino_Data))
                
                # 패킷 종료 데이터를 받은 경우
                if(ord(Arduino_Data)==0x03):
                    
                    # 수신 받은 데이터 파싱
                    if(recv_packet.parsingList(recv_list)):
                        
                        # 파싱한 데이터를 파싱완료 리스트에 저장
                        recv_parsing_packet.append(recv_packet)
                        
                        # 패킷 리스트 초기화
                        recv_list.clear()
                        
                        # 전송 완료 플래그 해제
                        send_flag=False
                        
                        # 반복문 탈출
                        break

In [22]:
def Received_packet():
    # 파싱 완료 리스트의 첫번째 패킷을 result 변수에 저장함
    result=recv_parsing_packet[0]
    
    # 저장 완료한 파싱 완료 리스트 삭제 
    del recv_parsing_packet[0]
    
    # result 변수값 반환
    return result

In [23]:
# 인자값으로 Received_packet() 함수에서 반환된 값을 넣어줌
def Ultrasonic_Data(packet):
    
    #전역 변수 사용
    global data_Ultrasonic_Sensors
    
    # 패킷에서 ID 추출
    packet_id=packet.getID()
    
    # 패킷에서 CMD 추출
    packet_cmd=packet.getCMD()
    
    # 추출한 ID 가 Arduino_ID 와 맞는지 확인
    if(packet_id==Arduino_ID):
        
        #추출한 CMD가 응답 CMD가 맞는지 확인
        if(packet_cmd==ANSWER_Ultra_Sonic_SENSOR):
            
            # 패킷에서 Payload 값을 추출함
            for payload in packet.getPayload():
                
                # 추출한 MID가 초음파 센서가 맞는지 확인
                if(payload.getID()==MID_Ultra_Sonic_SENSOR):
                    
                    # Payload 에서 읽어온 데이터를 버퍼에 저장 
                    buf=str(payload.getData())
                    
                    # 버퍼에 있는 데이터 값(초음파 센서 거리 값) 을 확인할수 있게 배열로 나누어 저장 
                    data_Ultrasonic_Sensors = [int(float(buf[:3])), 
                                               int(float(buf[3:6])),
                                               int(float(buf[6:9])), 
                                               int(float(buf[9:12])),
                                               int(float(buf[12:15])), 
                                               int(float(buf[15:]))]
                  

In [25]:
pca9685.frequency = melody[4]
lower_dist = 9999
while(True):
    
    # 데이터 요청 패킷 송신
    Packet_send(Arduino_ID, REQUEST_Ultra_Sonic_SENSOR, MID_Ultra_Sonic_SENSOR)
    
    # 응답 데이터 패킷 수신
    Packet_receive(Serial_Arduino)
    
    
    
    if(len(recv_parsing_packet) > 0):
        # 가장 가까운 거리 측정
        lower_dist = 9999
        
        # 파싱완료 리스트에서 첫번째 패킷을 가져옴
        p=Received_packet()
        
        # 수신한 패킷을 파싱하고 초음파 센서 거리값을 저장함
        Ultrasonic_Data(p)
        
        # 데이터 출력
        print("Ultra_Sonic : "+ str(data_Ultrasonic_Sensors))
        
        for d in data_Ultrasonic_Sensors:
            if(lower_dist > d):
                lower_dist = d
        
    
    if(lower_dist < 20):
        pca9685.channels[15].duty_cycle = int(4096/2)
        
        for mid in MID_list:

            # 모터 제어값을 아두이노에 전송합니다.
            Control_send(Arduino_ID, CONTROL_DRIVE, mid, 0)
        
    else:
        # MID_list 에 담겨있는 모터 MID 를 하나씩 가져옵니다.
#         for mid in MID_list:

        # 모터 제어값을 아두이노에 전송합니다.
        Control_send(Arduino_ID, CONTROL_DRIVE, MID_list[1], -300)
        pca9685.channels[15].duty_cycle = 0
        
    display_lcd(str(lower_dist))
    
    # 일정시간 대기
    time.sleep(0.5)

Ultra_Sonic : [77, 144, 65, 103, 194, 118]
Ultra_Sonic : [139, 7, 102, 64, 74, 155]
Ultra_Sonic : [145, 61, 100, 106, 138, 29]
Ultra_Sonic : [133, 70, 113, 81, 194, 12]
Ultra_Sonic : [135, 38, 60, 78, 198, 174]
Ultra_Sonic : [140, 71, 54, 89, 121, 201]


KeyboardInterrupt: 

In [26]:
Control_send(Arduino_ID, CONTROL_DRIVE, MID_list[0], 0)
Control_send(Arduino_ID, CONTROL_DRIVE, MID_list[1], 0)
Control_send(Arduino_ID, CONTROL_DRIVE, MID_list[2], 0)